## MetricHMSR: Metric Human Mesh and Scene Recovory from Monocular Images(CVPR'26)

### 1. Camera Model

- 3D공간에서 2D로 이미지로 변환하는 과정을 수학적으로 모델링하는 것

### 2. Pinhole Camera Model

- Pinhole Camera: 바늘구멍 카메라, 모든 빛이 하나의 점을 통과
- World coorinate($X_w, Y_w, Z_w$) $\rightarrow$ Camera coordinate($X_c, Y_c, Z_c$) $\rightarrow$ Image coordinate(u, v)
- World coordinate: 3D 공간의 절대 좌표
- Camera coordinate: 카메라를 원점으로 하는 좌표. Z축은 카메라가 바라보는 방향, X축이 오른쪽, Y축이 아래쪽
- Image coordinate: 2D 픽셀 좌표(u, v)

<img src='../assets/metric-1.png' height = 200 width = 500>


#### 1) Extrinsic Prameter

- $t \in \mathbb{R}^{3\times 1}$
$$X_c = R \cdot X_w + t$$
- translation 행렬 t: 모든 좌표의 기준점이 카메라 렌즈의 중심으로 옮겨짐
- rotation 행렬 R: 월드의 축들이 회전해 카메라 z,x,y로 옮겨짐

#### 2) Intrinsic Parameter: Calibration Matrix

$$K = \begin{pmatrix} f_x & 0 & c_x \\ 0 &f_y & c_y \\ 0 & 0 & 1 \end{pmatrix}$$
- $f_x, \: f_y$(focal length, 초점 거리): 카메라 렌즈(핀홀) 중심에서 이미지 이미지 센서까지의 초점 거리를 픽셀 단위로 표현한 것, 크면 zoom-in 효과, 작으면 wide-angle
- $c_x, \: c_y$(principal point, 주점): 이미지의 광학점 중심, 보통 이미지의 중심(W/2, H/2)
- $f_x, \: f_y, \: c_x, \: c_y$: Intrinsic parameters of camera, They represent the camera's internal geometry

#### 3) Perspective Projection: 3D(camera coordinate) $\rightarrow$ 2D(image coordinate)

$$X_i = \begin{pmatrix} u \\ v \\ 1 \end{pmatrix} = K \cdot \begin{pmatrix} x_c/z_c \\ y_c/z_c \\ z_c/z_c \end{pmatrix} = \begin{pmatrix} f_x & 0 & c_x \\ 0 &f_y & c_y \\ 0 & 0 & 1 \end{pmatrix}\begin{pmatrix} x_c/z_c \\ y_c/z_c \\ 1 \end{pmatrix}$$
$$u = f_x \cdot \frac{x_c}{z_c} + c_x$$
$$v = f_y \cdot \frac{y_c}{z_c} + c_y$$
- $Z$로 나누는 것: 원근법의 수학적 표현, 김피가 크면 이미지에서 작게 보이도록

#### 4) Backprojection: 2D $\rightarrow$ 3D (Ray Map)

- $X_i = \begin{pmatrix} u \\ v \\ 1 \end{pmatrix} = K \cdot X_c/z_c$
$$d = X_c/z_c = K^{-1}\begin{pmatrix} u \\ v \\ 1 \end{pmatrix}$$

- d: ray direction
- 각 픽셀에 $K^{-1}$을 곱해서 그 pixel을 통과하는 3D 방향 벡터를 얻음
- ray map $\mathbb{R}^{H \times W \times 3}$
$$K^{-1} = \begin{pmatrix} \frac{1}{f_x} & 0 & -\frac{c_x}{f_x} \\ 0 & \frac{1}{f_y} & -\frac{c_y}{f_y} \\ 0 & 0 & 1 \end{pmatrix}$$

5) weak vs full-perspective projection

- full-perspective projection
    - u = f · (X / Z) + cx
    - v = f · (Y / Z) + cy
- weak-perspective projection
    - u = s · X + tx
    - u = s · X + ty
- f/z = s

### 3. Crop과 위치 정보 손실 문제

#### 1) Top-down HMR의 crop 문제: HMR(CVPR'2018)
- HMR계열은 사람을 detect(YOLO, Faster R-CNN) $\rightarrow$ crop $\rightarrow$ 네트워크 입력 순서로 처리
- Crop을 하면 사람의 이미지가 어디에 있었는지 정보가 사라짐
- 같은 포즈인 왼쪽 끝 사람과 중앙의 사람의 crop은 동일하게 보이지만 실제 3D위치는 매우 다름
- $\hat{\mathbf{x}} = s \Pi(RX(\theta, \beta)) + t, \:$ whete $\Pi$ is an 2D projection
    - weak-perspective projection
- $\rightarrow$ s가 '크기 x 거리'를 동시에 담고 있어서 분리가 불가능, 원근 왜곡이 무시됨, 깊이 차이에 따른 크기 변화를 계산하지 않고 똑같은 비율로 누름

#### 2) CLIFF(2022)의 해결책 및 한계

<img src ='../assets/metric-2.png'>

- bbox 좌표($c_x, c_y, b$)- center x, center y, bbox size b를 focal length로 정규화 후 feature에 concat함
    - bbox info를 통해 위치정보를 반영해 왜곡을 학습
    - focal length로 나눠준 이유: 카메라와 무관하게 같은 3D 위치에 있는 사람은 같은 feature를 받게 하기 위해서

$$I_{bbox} = [\frac{c_x}{f_{CLIFF}}, \frac{c_y}{f_{CLIFF}}, \frac{b}{f_{CLIFF}}]$$

- HMR과 다르게 Loss를 full 이미지에 대해서 계산함
    - crop 안에서의 projection은 perspective distortion(원근 왜곡)을 왜곡
        - full frame에 대해 loss를 계산하면 실제 카메라의 perspective distortion을 그대로 반영
    - Full-frame Supervision의 효과
        - crop에서는 사람이 항상 중앙이여서 GT와의 오차가 아주 미세함. global translation을 정확히 학습해야할 동기(gradient)가 부족, 하지만 full-frame에서는 조금만 위치를 잘못하면 큰 loss가 발생하게 되어 global translation을 정확히 예측하도록 학습됨
- crop 좌표계의 translation도 full-frame으로 변환

<img src = '../assets/metric-3.png'>

- HMR의 scale대신 카메라로부터의 depth($t_z$)를 사용함
    - full-perspective projection

- 한계
    - 3개의 스칼라 값이 이미지 feature에 동일하게 concat됨, 여전히 픽셀별 차이를 표현할 수 없음 $\rightarrow$ 왜곡이 무시됨
    - focal length를 정규화에만 사용하기 때문에 bbox좌표만 보정하는 것, 각 픽셀의 ray direction을 계산하는 것이 아님
    - crop2full 변환이 후처리, 네트워크가 crop 좌표에서 먼저 예측하고 이후 수식으로 변환하기 때문에 정확성이 떨어짐
    - $\rightarrow$ 완벽한 metric scale 복원에는 실패함 

#### 3) Metric HMSR
- bbox 좌표 3개 대신, HxWx3 크기의 ray map을 만들어서넣음
- 각 픽셀에 대해서 원본 이미지 기준의 ray direction을 계산
- 픽셀마다 다른 정보 -> 픽셀마다 다른 ray direction을 계산해 spatial resolution을 유지, ray map을 통해 perspective distortion의 정도를 알 수 있음

### 4. MoE(Mixture of Experts)

<img src= '../assets/metric-4.png'>

#### 1) 일반 Transformer의 FFN
- Transformer block = Self-Attention + FFN
$$\text{FFN}(\mathbf{x}) = W_2 \sigma_\text{gelu}(W_1\mathbf{x} ) $$
- 모든 token이 같은 FFN을 공유

<img src= '../assets/metric-5.png'>

#### 2) MoE의 기본 구조
- FFN 하나 대신 여러 개의 "Expert" FFN을 배치
- Router(게이트 네트워크)가 각 token을 어떤 expert로 보낼지 결정
$$\text{MoE}(\mathbf{x}) = \sum^R_{i=1}g(\mathbf{x})_i e_i(\mathbf{x})$$
- $\mathbf{x} \in \mathbb{R}^D$
- $e_i$: expert $i$의 funtion $\mathbb{R}^D \mapsto\mathbb{R}^D$
- $e_i(\mathbf{x}) = \text{MLP}_{\theta_i}(\mathbf{x})$
- $g$: expert에 대한 가중치를 정하는 function
- $g(\mathbf{x}) = \text{TOP}_k(\text{softmax}(\mathbf{W}\mathbf{x}+ \epsilon))$
- 핵심: 서로 다른 종류의 token이 서로 다른 expert(FFN)로 가면서 자연스럽게 specialization이 일어남
- 
#### 3) Load Balancing Loss
- 문제: 학습 초기에 특정 expert가 우연히 좋은 결과를 내면, router가 그 expert로 점점 더 많은 token을 보냄, 나머지 expert는 학습이 되지 않음
- 몇몇 expert만 사용되고 대부분의 expert가 사용되지 않음
- $\rightarrow$ load balancing loss를 추가해 token이 expert들에 골고루 분배되도록 유도
- 해결: auxiliary loss로 expert 간 token 배분을 균등하게 유도


### 4. Camera HMR(3DV'2025)